Deep learning pipeline for video anomaly detection using PyTorch (VideoAnomalyDetection.ipynb v0.1)

*   Extracting frames from videos
*   Learning temporal/spatial patterns
*   Detecting abnormal events (e.g., unusual motion, behavior)




In [ ]:
# 1. Mount Google Drive (your dataset must be here)
from google.colab import drive
drive.mount('/content/drive')

# 2. Install requirements
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers opencv-python-headless scikit-learn matplotlib seaborn pandas

print("✅ Installed! Use Runtime → Change runtime type → GPU")

In [ ]:
# Handles evaluation and training
# Write metrics_manager.py to file
#
%%writefile metrics_manager.py

import time
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, auc, precision_score, recall_score, f1_score


class MetricsManager:
    def __init__(self, model_name):
        # Create a safe filename safe string (e.g. "3D CNN" -> "3D_CNN")
        self.cache_dir = 'cache'
        os.makedirs(self.cache_dir, exist_ok=True)
        self.safe_name = model_name.replace(" ", "_")
        self.metrics_file = os.path.join(self.cache_dir, f"metrics_{self.safe_name}.json")
        self.cm_file = os.path.join(self.cache_dir, f"cm_{self.safe_name}.png")
        self.roc_file = os.path.join(self.cache_dir, f"roc_{self.safe_name}.png")
        self.start_time = None
        self.end_time = None

    def start_training_timer(self):
        self.start_time = time.time()

    def stop_training_timer(self):
        self.end_time = time.time()
        return self.end_time - self.start_time

    def compute_metrics(self, y_true, y_pred, y_probs):
        """Calculates a comprehensive dictionary of metrics."""

        # 1. Confusion Matrix elements
        cm = confusion_matrix(y_true, y_pred)

        # Check if binary classification
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        else:
            # For multi-class, specificity isn't directly applicable
            specificity = None

        # 2. Derived Metrics
        accuracy = accuracy_score(y_true, y_pred)
        sensitivity = recall_score(y_true, y_pred, average='binary' if cm.shape == (2, 2) else 'macro', zero_division=0)
        precision = precision_score(y_true, y_pred, average='binary' if cm.shape == (2, 2) else 'macro', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='binary' if cm.shape == (2, 2) else 'macro', zero_division=0)

        # 3. AUC calculation
        try:
            fpr, tpr, _ = roc_curve(y_true, y_probs)
            roc_auc = auc(fpr, tpr)
        except:
            roc_auc = 0.5
            fpr, tpr = [0, 1], [0, 1]

        return {
            "accuracy": accuracy,
            "sensitivity": sensitivity,
            "specificity": specificity,
            "precision": precision,
            "f1_score": f1,
            "auc": roc_auc,
            "confusion_matrix": cm.tolist(),  # Convert to list for JSON
            "roc_data": {"fpr": fpr.tolist(), "tpr": tpr.tolist()}
        }

    def save_metrics(self, metrics_dict, training_time):
        """Saves metrics and training time to JSON."""
        metrics_dict["training_time_seconds"] = training_time
        metrics_dict["training_timestamp"] = time.ctime()

        with open(self.metrics_file, "w") as f:
            json.dump(metrics_dict, f, indent=4)
        print(f"Metrics saved to {self.metrics_file}")

    def load_metrics(self):
        """Loads the JSON metrics from disk."""
        if os.path.exists(self.metrics_file):
            with open(self.metrics_file, "r") as f:
                return json.load(f)
        return None

    def plot_confusion_matrix(self, cm):
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Normal', 'Abnormal'],
                    yticklabels=['Normal', 'Abnormal'])
        plt.title(f'Confusion Matrix: {self.safe_name}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig(self.cm_file)
        plt.close()

    def plot_roc_curve(self, fpr, tpr, roc_auc):
        plt.figure(figsize=(5, 4))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve: {self.safe_name}')
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(self.roc_file)
        plt.close()

In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
from torchvision.models import ResNet18_Weights

# Try importing Transformers (Handle case if user hasn't installed it)
try:
    from transformers import VideoMAEForVideoClassification

    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False


# ==========================================
# 1. CNN-LSTM (The Baseline)
# ==========================================
class CNNLSTM(nn.Module):
    def __init__(self, num_classes=2, lstm_hidden_size=256, lstm_layers=2):
        super(CNNLSTM, self).__init__()
        resnet = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        self.lstm = nn.LSTM(input_size=512, hidden_size=lstm_hidden_size, num_layers=lstm_layers, batch_first=True)
        self.fc = nn.Linear(lstm_hidden_size, num_classes)

    def forward(self, x):
        # Input: (Batch, Seq, C, H, W)
        batch_size, seq_len, c, h, w = x.size()
        c_in = x.view(batch_size * seq_len, c, h, w)
        cnn_out = self.feature_extractor(c_in)
        cnn_out = cnn_out.view(batch_size, seq_len, -1)
        lstm_out, _ = self.lstm(cnn_out)
        return self.fc(lstm_out[:, -1, :])


# ==========================================
# 2. 3D CNN (ResNet R(2+1)D)
# ==========================================
class ResNet3D(nn.Module):
    def __init__(self, num_classes=2):
        super(ResNet3D, self).__init__()
        weights = R2Plus1D_18_Weights.DEFAULT
        self.model = r2plus1d_18(weights=weights)
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        # Input: (Batch, Seq, C, H, W) -> Needed: (Batch, C, Seq, H, W)
        x = x.permute(0, 2, 1, 3, 4)
        return self.model(x)


# ==========================================
# 3. Video Transformer (VideoMAE)
# ==========================================
class VideoTransformer(nn.Module):
    def __init__(self, num_classes=2):
        super(VideoTransformer, self).__init__()
        if not TRANSFORMERS_AVAILABLE:
            raise ImportError("Please run 'pip install transformers' to use this model.")

        # Using a lightweight VideoMAE model
        self.model = VideoMAEForVideoClassification.from_pretrained(
            "MCG-NJU/videomae-base-finetuned-kinetics",
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )

    def forward(self, x):
        # Input: (Batch, Seq, C, H, W)
        # HuggingFace expects 'pixel_values' argument
        outputs = self.model(pixel_values=x)
        return outputs.logits


# ==========================================
# Factory Helper
# ==========================================
def get_model(model_name, num_classes=2):
    if model_name == "CNN-LSTM":
        return CNNLSTM(num_classes=num_classes)
    elif model_name == "3D CNN":
        return ResNet3D(num_classes=num_classes)
    elif model_name == "Video Transformer":
        return VideoTransformer(num_classes=num_classes)
    else:
        raise ValueError(f"Unknown model name: {model_name}")

In [ ]:
%%writefile train.py

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import cv2
import glob
import os
import numpy as np
from model import get_model
from metrics_manager import MetricsManager
import argparse

# --- CONFIG (updated for Colab / Google Drive) ---
IMG_SIZE = 224
SEQ_LEN = 16
BATCH_SIZE = 8
DEFAULT_ROOT = "/content/drive/MyDrive/VideoDataset"   # Updated dataset root

class VideoDataset(Dataset):
    def __init__(self, root_dir=DEFAULT_ROOT, dataset_name='crime-ucf'):
        self.video_paths = []
        self.labels = []
        # Adjusted glob.glob paths to include 'crime-ucf' subfolder
        norm_files = glob.glob(os.path.join(root_dir, dataset_name, 'normal', '*.mp4'))
        abnorm_files = glob.glob(os.path.join(root_dir, dataset_name, 'abnormal', '*.mp4'))
        self.video_paths.extend(norm_files + abnorm_files)
        self.labels.extend([0] * len(norm_files) + [1] * len(abnorm_files))

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        cap = cv2.VideoCapture(video_path)
        frames = []
        while len(frames) < SEQ_LEN:
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = frame.astype(np.float32) / 255.0
                frame = np.transpose(frame, (2, 0, 1))
                frames.append(frame)
            else:
                frames.append(np.zeros((3, IMG_SIZE, IMG_SIZE), dtype=np.float32))
        cap.release()
        return torch.tensor(np.array(frames[:SEQ_LEN])), torch.tensor(label)


def run_training(model_name, epochs=5, dataset_path=None, dataset_name='crime-ucf'):
    if dataset_path is None:
        dataset_path = DEFAULT_ROOT
    metrics_manager = MetricsManager(model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✅ Training on {device}")

    # Adjusted existence check for 'crime-ucf' subfolder
    if not os.path.exists(os.path.join(dataset_path, dataset_name)):
        return f"Error: Dataset not found at {os.path.join(dataset_path, dataset_name)}"

    full_dataset = VideoDataset(root_dir=dataset_path, dataset_name=dataset_name)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = get_model(model_name).to(device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    metrics_manager.start_training_timer()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for videos, labels in train_loader:
            videos, labels = videos.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(videos)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {running_loss / len(train_loader):.4f}")

    total_time = metrics_manager.stop_training_timer()

    # Save model
    os.makedirs('cache', exist_ok=True)
    safe_name = model_name.replace(" ", "_")
    save_path = f"cache/model_{safe_name}.pth"
    torch.save(model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(), save_path)

    # Evaluation & metrics (on validation set)
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for videos, labels in val_loader:
            videos, labels = videos.to(device), labels.to(device)
            outputs = model(videos)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            all_probs.extend(probs[:, 1].cpu().numpy())
            _, preds = torch.max(probs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    stats = metrics_manager.compute_metrics(all_labels, all_preds, all_probs)
    stats['epochs'] = epochs # Add epochs to stats
    metrics_manager.save_metrics(stats, total_time)
    metrics_manager.plot_confusion_matrix(np.array(stats['confusion_matrix']))
    metrics_manager.plot_roc_curve(stats['roc_data']['fpr'], stats['roc_data']['tpr'], stats['auc'])

    print(f"✅ Training complete! Model saved: {save_path}")
    return save_path


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Train video anomaly detection model.')
    parser.add_argument('--model', type=str, default='3D CNN',
                        choices=['CNN-LSTM', '3D CNN', 'Video Transformer'],
                        help='Model name to train (e.g., "3D CNN")')
    parser.add_argument('--epochs', type=int, default=5, help='Number of training epochs.')
    parser.add_argument('--dataset_path', type=str, default=DEFAULT_ROOT,
                        help='Path to the video dataset root directory.')
    parser.add_argument('--dataset_name', type=str, default='crime-ucf',
                        help='Name of the dataset subfolder (e.g., "crime-ucf", "kvasir").')

    args = parser.parse_args()

    run_training(args.model, args.epochs, args.dataset_path, args.dataset_name)


In [ ]:
# Command line interface for experiments
# Write app.py to file
#
%%writefile app.py

from metrics_manager import MetricsManager
import json
from IPython.display import Image, display
import os

# Assuming the last trained model was '3D CNN'
model_name = "3D CNN"
metrics_manager = MetricsManager(model_name)

# Load the saved metrics
metrics = metrics_manager.load_metrics()

if metrics:
    print(f"--- Metrics for {model_name} ---")
    print(f"Accuracy: {metrics.get('accuracy', 'N/A'):.4f}")
    print(f"Sensitivity: {metrics.get('sensitivity', 'N/A'):.4f}")
    print(f"Specificity: {metrics.get('specificity', 'N/A'):.4f}")
    print(f"Precision: {metrics.get('precision', 'N/A'):.4f}")
    print(f"F1 Score: {metrics.get('f1_score', 'N/A'):.4f}")
    print(f"AUC: {metrics.get('auc', 'N/A'):.4f}")
    print(f"Training Time: {metrics.get('training_time_seconds', 'N/A'):.2f} seconds")
    print(f"Training Timestamp: {metrics.get('training_timestamp', 'N/A')}")

    # Display confusion matrix and ROC curve images
    cm_path = f"cache/cm_{metrics_manager.safe_name}.png"
    roc_path = f"cache/roc_{metrics_manager.safe_name}.png"

    if os.path.exists(cm_path):
        print("\n--- Confusion Matrix ---")
        display(Image(filename=cm_path))
    else:
        print(f"Confusion Matrix image not found at {cm_path}")

    if os.path.exists(roc_path):
        print("\n--- ROC Curve ---")
        display(Image(filename=roc_path))
    else:
        print(f"ROC Curve image not found at {roc_path}")
else:
    print(f"No metrics found for {model_name}. Please ensure the model has been trained and metrics saved.")

In [ ]:
# 1. Train the '3D CNN' model
!python train.py --model "3D CNN" --epochs 5 --dataset_path "/content/drive/MyDrive/VideoDataset" --dataset_name "kvasir"

# 2. Train the 'CNN-LSTM' model
!python train.py --model "CNN-LSTM" --epochs 5 --dataset_path "/content/drive/MyDrive/VideoDataset" --dataset_name "kvasir"

# 3. Train the 'Video Transformer' model
!python train.py --model "Video Transformer" --epochs 5 --dataset_path "/content/drive/MyDrive/VideoDataset" --dataset_name "kvasir"

In [ ]:
%%writefile evaluate_model.py

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cv2
import glob
import os
import numpy as np
from model import get_model # Assuming model.py is available
from metrics_manager import MetricsManager # Assuming metrics_manager.py is available
import argparse
from IPython.display import Image, display

# --- CONFIG ---
IMG_SIZE = 224
SEQ_LEN = 16
BATCH_SIZE = 8
DEFAULT_ROOT = "/content/drive/MyDrive/VideoDataset"

class VideoDataset(Dataset):
    def __init__(self, root_dir=DEFAULT_ROOT, dataset_name='crime-ucf'):
        self.video_paths = []
        self.labels = []
        # Adjusted glob.glob paths to include dataset_name subfolder
        norm_files = glob.glob(os.path.join(root_dir, dataset_name, 'normal', '*.mp4'))
        abnorm_files = glob.glob(os.path.join(root_dir, dataset_name, 'abnormal', '*.mp4'))
        self.video_paths.extend(norm_files + abnorm_files)
        self.labels.extend([0] * len(norm_files) + [1] * len(abnorm_files))

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        cap = cv2.VideoCapture(video_path)
        frames = []
        while len(frames) < SEQ_LEN:
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = frame.astype(np.float32) / 255.0
                frame = np.transpose(frame, (2, 0, 1))
                frames.append(frame)
            else:
                # Pad with black frames if video is too short
                frames.append(np.zeros((3, IMG_SIZE, IMG_SIZE), dtype=np.float32))
        cap.release()
        return torch.tensor(np.array(frames[:SEQ_LEN])), torch.tensor(label)


def run_evaluation(model_name, model_path, epochs, dataset_path=None, dataset_name='crime-ucf'):
    if dataset_path is None:
        dataset_path = DEFAULT_ROOT

    # Create a unique name for metrics for evaluation runs, e.g., "3D_CNN_eval_crime-ucf"
    eval_model_name = f"{model_name.replace(' ', '_')}_eval_{dataset_name}"
    metrics_manager = MetricsManager(eval_model_name)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"✅ Evaluating {model_name} on {dataset_name} using {device}")

    # Check dataset existence
    if not os.path.exists(os.path.join(dataset_path, dataset_name)):
        print(f"Error: Dataset not found at {os.path.join(dataset_path, dataset_name)}")
        return

    # Load dataset for evaluation
    evaluation_dataset = VideoDataset(root_dir=dataset_path, dataset_name=dataset_name)
    eval_loader = DataLoader(evaluation_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Load model
    model = get_model(model_name).to(device)
    state_dict = torch.load(model_path, map_location=device)
    # Handle DataParallel saved state dict
    if list(state_dict.keys())[0].startswith('module.'):
        state_dict = {k[7:]: v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.eval() # Set model to evaluation mode

    # Perform evaluation
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for videos, labels in eval_loader:
            videos, labels = videos.to(device), labels.to(device)
            outputs = model(videos)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            all_probs.extend(probs[:, 1].cpu().numpy())
            _, preds = torch.max(probs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Compute and save metrics
    stats = metrics_manager.compute_metrics(all_labels, all_preds, all_probs)
    stats['epochs'] = epochs # Add epochs to stats
    metrics_manager.save_metrics(stats, 0) # Training time is 0 for evaluation
    metrics_manager.plot_confusion_matrix(np.array(stats['confusion_matrix']))
    metrics_manager.plot_roc_curve(stats['roc_data']['fpr'], stats['roc_data']['tpr'], stats['auc'])

    print(f"✅ Evaluation complete for {model_name} on {dataset_name}.")
    display_metrics(eval_model_name) # Display metrics within the notebook context


def display_metrics(model_identifier):
    metrics_manager = MetricsManager(model_identifier)
    metrics = metrics_manager.load_metrics()

    if metrics:
        print(f"\n--- Metrics for {model_identifier} ---")
        print(f"Accuracy: {metrics.get('accuracy', 'N/A'):.4f}")
        print(f"Sensitivity: {metrics.get('sensitivity', 'N/A'):.4f}")
        print(f"Specificity: {metrics.get('specificity', 'N/A'):.4f}")
        print(f"Precision: {metrics.get('precision', 'N/A'):.4f}")
        print(f"F1 Score: {metrics.get('f1_score', 'N/A'):.4f}")
        print(f"AUC: {metrics.get('auc', 'N/A'):.4f}")
        print(f"Epochs: {metrics.get('epochs', 'N/A')}") # Display epochs
        print(f"Training Time: {metrics.get('training_time_seconds', 'N/A'):.2f} seconds (0 for evaluation)")
        print(f"Training Timestamp: {metrics.get('training_timestamp', 'N/A')}")

        cm_path = f"cache/cm_{metrics_manager.safe_name}.png"
        roc_path = f"cache/roc_{metrics_manager.safe_name}.png"

        if os.path.exists(cm_path):
            print("\n--- Confusion Matrix ---")
            display(Image(filename=cm_path))
        else:
            print(f"Confusion Matrix image not found at {cm_path}")

        if os.path.exists(roc_path):
            print("\n--- ROC Curve ---")
            display(Image(filename=roc_path))
        else:
            print(f"ROC Curve image not found at {roc_path}")
    else:
        print(f"No metrics found for {model_identifier}.")

if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Evaluate a trained video anomaly detection model.')
    parser.add_argument('--model', type=str, required=True,
                        choices=['CNN-LSTM', '3D CNN', 'Video Transformer'],
                        help='Model name for evaluation (e.g., "3D CNN")')
    parser.add_argument('--model_path', type=str, required=True,
                        help='Path to the trained model weights (e.g., "cache/model_3D_CNN.pth")')
    parser.add_argument('--epochs', type=int, required=True,
                        help='Number of epochs the model was trained for.') # New argument
    parser.add_argument('--dataset_path', type=str, default=DEFAULT_ROOT,
                        help='Path to the video dataset root directory.')
    parser.add_argument('--dataset_name', type=str, required=True,
                        help='Name of the dataset subfolder to evaluate on (e.g., "crime-ucf", "kvasir").')

    args = parser.parse_args()

    run_evaluation(args.model, args.model_path, args.epochs, args.dataset_path, args.dataset_name)


In [ ]:
# Example usage for the new evaluate_model.py script

# Evaluate the '3D CNN' model on the 'crime-ucf' dataset
!python evaluate_model.py --model "3D CNN" \
  --model_path "cache/model_3D_CNN.pth" \
  --epochs 5 \
  --dataset_path "/content/drive/MyDrive/VideoDataset" \
  --dataset_name "kvasir"

# Evaluate the 'CNN-LSTM' model on the 'crime-ucf' dataset
!python evaluate_model.py --model "CNN-LSTM" \
  --model_path "cache/model_CNN-LSTM.pth" \
  --epochs 5 \
  --dataset_path "/content/drive/MyDrive/VideoDataset" \
  --dataset_name "kvasir"

 # Evaluate the 'Video Transformer' model on the 'crime-ucf' dataset
!python evaluate_model.py --model "Video Transformer" \
  --model_path "cache/model_Video_Transformer.pth" \
  --epochs 5 \
  --dataset_path "/content/drive/MyDrive/VideoDataset" \
  --dataset_name "kvasir"

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import numpy as np

# Helper function to load metrics
def load_metrics_for_comparison(model_identifier):
    cache_dir = 'cache'
    safe_name = model_identifier.replace(" ", "_")
    metrics_file = os.path.join(cache_dir, f"metrics_{safe_name}.json")
    if os.path.exists(metrics_file):
        with open(metrics_file, "r") as f:
            return json.load(f)
    return None

# Load metrics for all three models (using the evaluation run identifiers)
cnn_lstm_metrics = load_metrics_for_comparison("CNN-LSTM_eval_kvasir")
cnn3d_metrics = load_metrics_for_comparison("3D CNN_eval_kvasir")
video_transformer_metrics = load_metrics_for_comparison("Video Transformer_eval_kvasir")

# Dynamically retrieve epochs from metrics, default to 3 if not found (based on train.py calls)
epochs_cnn_lstm = cnn_lstm_metrics.get('epochs', 3) if cnn_lstm_metrics else 3
epochs_3dcnn = cnn3d_metrics.get('epochs', 3) if cnn3d_metrics else 3
epochs_video_transformer = video_transformer_metrics.get('epochs', 3) if video_transformer_metrics else 3

# Assuming all models were trained for the same number of epochs for this comparison
common_epochs = epochs_cnn_lstm # Or any of the epoch variables

# --- Plotting ROC Curves side-by-side (2x2 layout) ---
fig, axes = plt.subplots(2, 2, figsize=(18, 10)) # Changed to 2x2 subplots
fig.suptitle(f'ROC Curves Comparison on Kvasir Dataset ({common_epochs} Epochs)', fontsize=16) # Main title with epochs

# Flatten axes for easier indexing
axes = axes.flatten()

if cnn_lstm_metrics and 'roc_data' in cnn_lstm_metrics:
    fpr_cnn_lstm = np.array(cnn_lstm_metrics['roc_data']['fpr'])
    tpr_cnn_lstm = np.array(cnn_lstm_metrics['roc_data']['tpr'])
    auc_cnn_lstm = cnn_lstm_metrics['auc']
    axes[0].plot(fpr_cnn_lstm, tpr_cnn_lstm, color='blue', lw=2)
    axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[0].set_xlim([0.0, 1.0])
    axes[0].set_ylim([0.0, 1.05])
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title(f'CNN-LSTM (AUC = {auc_cnn_lstm:.2f})') # Simplified title with AUC
    axes[0].grid(True)
else:
    print("CNN-LSTM ROC data not found.")

if cnn3d_metrics and 'roc_data' in cnn3d_metrics:
    fpr_3dcnn = np.array(cnn3d_metrics['roc_data']['fpr'])
    tpr_3dcnn = np.array(cnn3d_metrics['roc_data']['tpr'])
    auc_3dcnn = cnn3d_metrics['auc']
    axes[1].plot(fpr_3dcnn, tpr_3dcnn, color='red', lw=2)
    axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[1].set_xlim([0.0, 1.0])
    axes[1].set_ylim([0.0, 1.05])
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'3D CNN (AUC = {auc_3dcnn:.2f})') # Simplified title with AUC
    axes[1].grid(True)
else:
    print("3D CNN ROC data not found.")

if video_transformer_metrics and 'roc_data' in video_transformer_metrics:
    fpr_video_transformer = np.array(video_transformer_metrics['roc_data']['fpr'])
    tpr_video_transformer = np.array(video_transformer_metrics['roc_data']['tpr'])
    auc_video_transformer = video_transformer_metrics['auc']
    axes[2].plot(fpr_video_transformer, tpr_video_transformer, color='green', lw=2)
    axes[2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[2].set_xlim([0.0, 1.0])
    axes[2].set_ylim([0.0, 1.05])
    axes[2].set_xlabel('False Positive Rate')
    axes[2].set_ylabel('True Positive Rate')
    axes[2].set_title(f'Video Transformer (AUC = {auc_video_transformer:.2f})') # Simplified title with AUC
    axes[2].grid(True)
else:
    print("Video Transformer ROC data not found.")

# Hide the fourth subplot since we only have three models
axes[3].set_visible(False)

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap
plt.show()

# --- Plotting Confusion Matrices side-by-side (2x2 layout) ---
fig, axes = plt.subplots(2, 2, figsize=(18, 10)) # Changed to 2x2 subplots
fig.suptitle(f'Confusion Matrices Comparison on Kvasir Dataset ({common_epochs} Epochs)', fontsize=16) # Main title with epochs

# Flatten axes for easier indexing
axes = axes.flatten()

if cnn_lstm_metrics and 'confusion_matrix' in cnn_lstm_metrics:
    cm_cnn_lstm = np.array(cnn_lstm_metrics['confusion_matrix'])
    sns.heatmap(cm_cnn_lstm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Abnormal'], yticklabels=['Normal', 'Abnormal'], ax=axes[0])
    axes[0].set_title('CNN-LSTM') # Simplified title
    axes[0].set_ylabel('True Label')
    axes[0].set_xlabel('Predicted Label')
else:
    print("CNN-LSTM Confusion Matrix data not found.")

if cnn3d_metrics and 'confusion_matrix' in cnn3d_metrics:
    cm_3dcnn = np.array(cnn3d_metrics['confusion_matrix'])
    sns.heatmap(cm_3dcnn, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Abnormal'], yticklabels=['Normal', 'Abnormal'], ax=axes[1])
    axes[1].set_title('3D CNN') # Simplified title
    axes[1].set_ylabel('True Label')
    axes[1].set_xlabel('Predicted Label')
else:
    print("3D CNN Confusion Matrix data not found.")

if video_transformer_metrics and 'confusion_matrix' in video_transformer_metrics:
    cm_video_transformer = np.array(video_transformer_metrics['confusion_matrix'])
    sns.heatmap(cm_video_transformer, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Abnormal'], yticklabels=['Normal', 'Abnormal'], ax=axes[2])
    axes[2].set_title('Video Transformer') # Simplified title
    axes[2].set_ylabel('True Label')
    axes[2].set_xlabel('Predicted Label')
else:
    print("Video Transformer Confusion Matrix data not found.")

# Hide the fourth subplot
axes[3].set_visible(False)

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap
plt.show()